# Cross-Venue Market Mapping

Goal: build a curated set of 8 binary prediction markets that are listed on **both** Kalshi and Polymarket, spanning 4 categories (Macro/Fed, Crypto, Politics, Sports).

Methodology: agent proposes candidate pairings per category; user approves each before it lands in `markets.yaml`. Manual curation beats fuzzy text matching because question wording diverges sharply between venues (Polymarket is conversational, Kalshi is formal).

Output: `markets.yaml` with 8 approved entries.

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, "../src")

from pm_micro.clients import kalshi, polymarket
import json

def show_pair(category, kalshi_market, polymarket_market):
    """Pretty-print a candidate pair for user review."""
    print(f"\n{'='*80}")
    print(f"CATEGORY: {category}")
    print(f"{'='*80}")
    print(f"\n  KALSHI:")
    print(f"    ticker: {kalshi_market['ticker']}")
    print(f"    title:  {kalshi_market.get('title', '?')}")
    print(f"    yes_sub: {kalshi_market.get('yes_sub_title', '?')}")
    print(f"    volume: {kalshi_market.get('volume', 0)}")
    print(f"    close:  {kalshi_market.get('close_time', '?')}")
    print(f"\n  POLYMARKET:")
    print(f"    condition_id: {polymarket_market['condition_id']}")
    print(f"    question:     {polymarket_market['question']}")
    print(f"    volume:       {polymarket_market.get('volume', 0)}")
    print(f"    end_date:     {polymarket_market.get('endDate', '?')}")
    token_ids = polymarket_market.get('clobTokenIds')
    if isinstance(token_ids, str):
        token_ids = json.loads(token_ids)
    print(f"    yes_token:    {token_ids[0] if token_ids else '?'}")
    print(f"    no_token:     {token_ids[1] if token_ids and len(token_ids) > 1 else '?'}")
    print()

## Category 1 — Macro / Fed

We already validated `FEDHIKE-26DEC31` in Phase 1. Find Polymarket's equivalent (Fed rate decision by end of 2026).

In [ ]:
# Kalshi candidate (already known from Phase 1)
kalshi_fed = next((m for m in kalshi.search_markets(series_ticker="KXFEDHIKE", status="open")
                   if m['ticker'] == "FEDHIKE-26DEC31"), None)

# Polymarket candidates
poly_fed_candidates = polymarket.search_markets("fed", limit=50)
print(f"Found {len(poly_fed_candidates)} Polymarket markets mentioning 'fed'")
for i, m in enumerate(poly_fed_candidates[:15]):
    print(f"  [{i}] {m['question'][:80]} | vol={m.get('volume', 0):.0f}")

In [ ]:
# ↓ User: pick the index of the best Polymarket match (or set to None if no match)
# Look for one matching "Fed hike by end of 2026" or close equivalent
POLY_FED_INDEX = None  # <-- USER FILLS THIS IN

if POLY_FED_INDEX is not None and kalshi_fed:
    show_pair("Macro/Fed", kalshi_fed, poly_fed_candidates[POLY_FED_INDEX])
    print("✅ If this looks right, copy the values into markets.yaml entry 1 below.")
    print("❌ If the questions don't actually match, set POLY_FED_INDEX = None and search again.")
elif kalshi_fed:
    print("⚠ Kalshi market found but no Polymarket match selected yet.")
    print("Run search_markets with different queries: 'rate', 'interest rate', 'federal reserve'")
else:
    print("⚠ Kalshi market not found — check series_ticker.")

## Category 2 — Crypto

Bitcoin price level markets exist on both venues. Find a matching pair (e.g. "BTC > $100k by Dec 2026").

In [ ]:
# Kalshi: search across crypto-related series
kalshi_btc_candidates = kalshi.search_markets(series_ticker="KXBTC", status="open", limit=50)
print(f"Found {len(kalshi_btc_candidates)} Kalshi KXBTC markets")
for i, m in enumerate(kalshi_btc_candidates[:15]):
    print(f"  [{i}] {m['ticker']:30s} | {m.get('title', '')[:60]} | vol={m.get('volume', 0)}")

# Try alternate series if KXBTC is empty
if not kalshi_btc_candidates:
    print("\nKXBTC empty — trying KXBTCD (daily Bitcoin):")
    kalshi_btc_candidates = kalshi.search_markets(series_ticker="KXBTCD", status="open", limit=20)
    for i, m in enumerate(kalshi_btc_candidates[:15]):
        print(f"  [{i}] {m['ticker']:30s} | {m.get('title', '')[:60]}")

In [ ]:
# Polymarket candidates
poly_btc_candidates = polymarket.search_markets("bitcoin", limit=50)
print(f"Found {len(poly_btc_candidates)} Polymarket markets mentioning 'bitcoin'")
for i, m in enumerate(poly_btc_candidates[:20]):
    print(f"  [{i}] {m['question'][:80]} | vol={m.get('volume', 0):.0f}")

In [ ]:
# ↓ User: pick indices for best Kalshi + Polymarket match (or None if no match)
KALSHI_BTC_INDEX = None
POLY_BTC_INDEX = None

if KALSHI_BTC_INDEX is not None and POLY_BTC_INDEX is not None:
    show_pair("Crypto", kalshi_btc_candidates[KALSHI_BTC_INDEX], poly_btc_candidates[POLY_BTC_INDEX])
else:
    print("⚠ Set both indices above to view the pair.")

## Category 3 — Politics

Candidates: 2026 Senate control, presidential approval thresholds, governorship races, midterm outcomes.

In [ ]:
# Kalshi: try several political series tickers
for series in ["KXSENATEM", "KXHOUSEM", "KXPRESAPPROVE", "KXGOVCONTROL"]:
    results = kalshi.search_markets(series_ticker=series, status="open", limit=10)
    if results:
        print(f"\n--- {series} ({len(results)} markets) ---")
        for i, m in enumerate(results[:5]):
            print(f"  [{i}] {m['ticker']:35s} | {m.get('title', '')[:60]}")
    else:
        print(f"--- {series}: no open markets")

# Also try unfiltered search and filter client-side
all_politics = kalshi.search_markets(status="open", limit=200, min_volume=10000)
politics_keywords = ["senate", "house", "election", "midterm", "president", "governor"]
print(f"\n--- Politics-keyword markets (from top 200 by status=open) ---")
for m in all_politics[:50]:
    title = m.get('title', '').lower()
    if any(kw in title for kw in politics_keywords):
        print(f"  {m['ticker']:35s} | {m.get('title', '')[:60]} | vol={m.get('volume', 0)}")

In [ ]:
poly_politics = []
for query in ["senate", "midterm", "election 2026", "president"]:
    results = polymarket.search_markets(query, limit=30)
    for r in results:
        if r not in poly_politics:
            poly_politics.append(r)
print(f"Found {len(poly_politics)} unique Polymarket political markets")
for i, m in enumerate(poly_politics[:20]):
    print(f"  [{i}] {m['question'][:80]} | vol={m.get('volume', 0):.0f}")

In [ ]:
# ↓ User: set indices for the politics pair
KALSHI_POLITICS_TICKER = None  # USER: paste full ticker as a string, e.g. "SENATEM-26NOV-..."
POLY_POLITICS_INDEX = None

if KALSHI_POLITICS_TICKER and POLY_POLITICS_INDEX is not None:
    kalshi_match = next((m for m in all_politics if m['ticker'] == KALSHI_POLITICS_TICKER), None)
    if kalshi_match:
        show_pair("Politics", kalshi_match, poly_politics[POLY_POLITICS_INDEX])
    else:
        print(f"⚠ Ticker {KALSHI_POLITICS_TICKER} not in current Kalshi results. Re-search.")

## Category 4 — Sports

World Cup 2026 winner is the strongest cross-venue candidate (Paradigm Predictions piece explicitly flags it). Also: Super Bowl winner, NBA Finals winner.

In [ ]:
# Kalshi sports series
for series in ["KXWORLDCUP", "KXNBACHAMP", "KXSBCHAMP", "KXNFLCHAMP"]:
    results = kalshi.search_markets(series_ticker=series, status="open", limit=15)
    if results:
        print(f"\n--- {series} ({len(results)} markets) ---")
        for i, m in enumerate(results[:10]):
            print(f"  [{i}] {m['ticker']:35s} | {m.get('title', '')[:50]} | vol={m.get('volume', 0)}")
    else:
        print(f"--- {series}: no open markets")

In [ ]:
poly_sports = []
for query in ["world cup", "nba", "super bowl", "champions league"]:
    results = polymarket.search_markets(query, limit=20)
    for r in results:
        if r not in poly_sports:
            poly_sports.append(r)
print(f"Found {len(poly_sports)} unique Polymarket sports markets")
for i, m in enumerate(poly_sports[:20]):
    print(f"  [{i}] {m['question'][:80]} | vol={m.get('volume', 0):.0f}")

In [ ]:
# ↓ User: paste full Kalshi ticker and Polymarket index for sports pair
KALSHI_SPORTS_TICKER = None
POLY_SPORTS_INDEX = None

if KALSHI_SPORTS_TICKER and POLY_SPORTS_INDEX is not None:
    # Re-fetch the specific market by ticker
    all_sports = []
    for series in ["KXWORLDCUP", "KXNBACHAMP", "KXSBCHAMP", "KXNFLCHAMP"]:
        all_sports.extend(kalshi.search_markets(series_ticker=series, status="open", limit=20))
    kalshi_match = next((m for m in all_sports if m['ticker'] == KALSHI_SPORTS_TICKER), None)
    if kalshi_match:
        show_pair("Sports", kalshi_match, poly_sports[POLY_SPORTS_INDEX])

## Second pair per category

For each category above, identify a *second* pairing to reach 8 total. Run the cells above with different indices/tickers. Document each pair below.

(User: add markdown cells with the chosen pairs for categories 1b, 2b, 3b, 4b)

## Final markets.yaml entries

Once all 8 pairs are approved, the entries below are ready to be written to `../markets.yaml`. Format:

```yaml
- id: macro_fed_dec2026
  category: macro_fed
  description: "Will the Fed hike rates by Dec 31, 2026?"
  kalshi:
    ticker: FEDHIKE-26DEC31
  polymarket:
    condition_id: "0x..."
    yes_token_id: "..."
    no_token_id: "..."
  match_notes: "Both markets resolve on Fed FOMC decisions by EOY 2026. Kalshi resolves on any hike action; Polymarket question requires reading rules to confirm same resolution criterion."
```

In [ ]:
# Run this LAST, after all 8 pairs are user-approved.
# This cell writes markets.yaml.

APPROVED_PAIRS = [
    # USER: fill this in after approving all 8 pairs above.
    # Format: list of dicts with id, category, description, kalshi.ticker,
    # polymarket.condition_id, polymarket.yes_token_id, polymarket.no_token_id, match_notes
]

if len(APPROVED_PAIRS) == 8:
    import yaml
    with open("../markets.yaml", "w") as f:
        yaml.dump(APPROVED_PAIRS, f, sort_keys=False, default_flow_style=False)
    print(f"✅ Wrote {len(APPROVED_PAIRS)} pairs to markets.yaml")
else:
    print(f"⚠ APPROVED_PAIRS has {len(APPROVED_PAIRS)} entries; need exactly 8.")

## Validation

After writing markets.yaml, the next cell verifies every entry resolves to a live orderbook on both venues.

In [ ]:
import yaml
with open("../markets.yaml") as f:
    markets = yaml.safe_load(f)

for m in markets:
    print(f"\n{m['id']}:")
    try:
        kbook = kalshi.get_orderbook(m['kalshi']['ticker'])
        print(f"  ✅ Kalshi {m['kalshi']['ticker']}: orderbook fetched")
    except Exception as e:
        print(f"  ❌ Kalshi {m['kalshi']['ticker']}: {e}")
    try:
        pbook = polymarket.get_orderbook(m['polymarket']['yes_token_id'])
        print(f"  ✅ Polymarket YES token: orderbook fetched (bids={len(pbook.bids)}, asks={len(pbook.asks)})")
    except Exception as e:
        print(f"  ❌ Polymarket: {e}")